# Patient Portal v2 — A100 Parallel Pipeline

Runs 3 modes on 2000 questions with `OLLAMA_NUM_PARALLEL=4` + `ThreadPoolExecutor`.

**Expected runtime on A100:** ~1.5–2 hours (vs ~8h sequential on T4).

**Outputs (in Drive):**
- `synthetic_patients/patient_portal_responses_v2_2000.jsonl` (2000 records, all 3 modes)
- `synthetic_patients/patient_portal_ground_truth_v2_2000.jsonl` (winning_mode labels)

**To retrain the router with these labels**, copy these two files back to your Mac and re-run `scripts/retrain_router_v3.py` locally.

## Cell 1: Setup — Ollama with parallelism, drive mount, deps

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!apt-get update -qq && apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time
env = os.environ.copy()
env["OLLAMA_NUM_PARALLEL"] = "4"        # 4 concurrent generations
env["OLLAMA_MAX_LOADED_MODELS"] = "1"
env["OLLAMA_KEEP_ALIVE"] = "60m"
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env=env,
)
time.sleep(8)
print("Ollama up with OLLAMA_NUM_PARALLEL=4")
!ollama pull gemma2
print("Gemma2 ready")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out
W: Failed to fetch https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user

In [3]:
import sys
need_restart = False
try:
    import numpy
    if int(numpy.__version__.split('.')[0]) >= 2:
        need_restart = True
except ImportError:
    pass

!pip install --quiet "numpy<2" "thinc<8.4" "spacy<3.8" scispacy
!pip install --quiet faiss-gpu-cu12 transformers tqdm
!pip install --quiet https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

if need_restart:
    print("\nNumpy was 2.x — restarting kernel. Re-run from cell 1.")
    import os
    os.kill(os.getpid(), 9)
else:
    print("Deps installed (numpy already <2)")

  Preparing metadata (setup.py) ... done
Deps installed (numpy already <2)


In [4]:
import json, pickle, re, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import numpy as np
import requests
import torch
import torch.nn.functional as F
import faiss
import spacy
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}  ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")

DRIVE = Path("/content/drive/MyDrive/DL Project")
DATA = DRIVE / "synthetic_patients"
INDEX_DIR = DRIVE / "patient_index_v2"
INDEX_DIR.mkdir(exist_ok=True)

OLLAMA_URL = "http://localhost:11434/api/generate"
GEMMA_MODEL = "gemma2"
MAX_WORKERS = 4   # match OLLAMA_NUM_PARALLEL

Device: cuda  (NVIDIA A100-SXM4-80GB)


## Cell 2: Load data

In [5]:
def load_jsonl(p):
    with open(p) as f:
        return [json.loads(l) for l in f if l.strip()]

patients = load_jsonl(DATA / "patients_v2.jsonl")
patient_by_id = {p["patient_id"]: p for p in patients}
train_q = load_jsonl(DATA / "questions_v2.jsonl")
test_q = load_jsonl(DATA / "test_questions_v2.jsonl")
all_questions = train_q + test_q
print(f"patients: {len(patients)}  train: {len(train_q)}  test: {len(test_q)}  total Q: {len(all_questions)}")

patients: 400  train: 1600  test: 400  total Q: 2000


## Cell 3: Load MedCPT, scispaCy, PrimeKG

In [6]:
models_dir = DRIVE / "models"
qtok = AutoTokenizer.from_pretrained(models_dir / "MedCPT-Query-Encoder")
qenc = AutoModel.from_pretrained(models_dir / "MedCPT-Query-Encoder").to(DEVICE).eval()
atok = AutoTokenizer.from_pretrained(models_dir / "MedCPT-Article-Encoder")
aenc = AutoModel.from_pretrained(models_dir / "MedCPT-Article-Encoder").to(DEVICE).eval()
ctok = AutoTokenizer.from_pretrained(models_dir / "MedCPT-Cross-Encoder")
cenc = AutoModelForSequenceClassification.from_pretrained(models_dir / "MedCPT-Cross-Encoder").to(DEVICE).eval()
print("MedCPT loaded")

nlp = spacy.load("en_ner_bc5cdr_md")
print("scispaCy loaded")

with open(DRIVE / "primekg_index.pkl", "rb") as f:
    name_to_triples = pickle.load(f)
print(f"PrimeKG: {len(name_to_triples):,} entities")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

MedCPT loaded


/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


scispaCy loaded
PrimeKG: 128,550 entities


## Cell 4: Build patient FAISS for 400 patients

In [7]:
def patient_to_chunks(p):
    pid, name, age, gender = p["patient_id"], p["name"], p["age"], p["gender"].lower()
    chunks = []
    conds = p.get("active_conditions", [])
    if conds:
        s = "; ".join(f"{c['condition']} (diagnosed {c['diagnosed']})" for c in conds)
        text = f"{name} is a {age}-year-old {gender} with the following active medical conditions: {s}."
    else:
        text = f"{name} is a {age}-year-old {gender} with no active medical conditions documented."
    chunks.append({"chunk_id": f"{pid}_dem_cond", "patient_id": pid, "section": "demographics_and_conditions", "text": text})
    pmh = p.get("past_medical_history", [])
    if pmh:
        chunks.append({"chunk_id": f"{pid}_pmh", "patient_id": pid, "section": "past_medical_history", "text": f"{name} has the following past medical and surgical history: " + "; ".join(pmh) + "."})
    al = p.get("allergies", [])
    if al:
        text = f"{name} has the following documented drug or substance allergies: " + "; ".join(f"{a['substance']} ({a.get('reaction','unspecified')})" for a in al) + "."
    else:
        text = f"{name} has no documented drug allergies."
    chunks.append({"chunk_id": f"{pid}_allergies", "patient_id": pid, "section": "allergies", "text": text})
    v = p.get("recent_vitals", {})
    if v:
        date = v.get("date", "recent visit")
        parts = [f"{k.replace('_',' ')} {val}" for k, val in v.items() if k != "date"]
        chunks.append({"chunk_id": f"{pid}_vitals", "patient_id": pid, "section": "recent_vitals", "text": f"{name}'s recent vitals from {date}: " + ", ".join(parts) + "."})
    ls = p.get("lifestyle", {})
    if ls:
        parts = [f"{k.replace('_',' ')}: {val}" for k, val in ls.items()]
        chunks.append({"chunk_id": f"{pid}_lifestyle", "patient_id": pid, "section": "lifestyle", "text": f"{name}'s lifestyle factors. " + ". ".join(parts) + "."})
    return chunks

all_chunks = []
for p in tqdm(patients, desc="chunking"):
    all_chunks.extend(patient_to_chunks(p))
print(f"Total chunks: {len(all_chunks)}")

def encode_batch(texts, tok, enc, batch_size=32, max_length=256):
    embs = []
    for i in range(0, len(texts), batch_size):
        b = texts[i:i+batch_size]
        e = tok(b, truncation=True, padding=True, max_length=max_length, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            o = enc(**e)
            v = F.normalize(o.last_hidden_state[:,0,:], dim=-1)
        embs.append(v.cpu().numpy())
    return np.vstack(embs)

chunk_embs = encode_batch([c["text"] for c in all_chunks], atok, aenc)
print(f"Embeddings: {chunk_embs.shape}")

patient_chunks_by_id, patient_embs_by_id = {}, {}
for c, e in zip(all_chunks, chunk_embs):
    patient_chunks_by_id.setdefault(c["patient_id"], []).append(c)
    patient_embs_by_id.setdefault(c["patient_id"], []).append(e)
for pid in patient_embs_by_id:
    patient_embs_by_id[pid] = np.array(patient_embs_by_id[pid])
print(f"Per-patient embeddings ready for {len(patient_embs_by_id)} patients")

chunking:   0%|          | 0/400 [00:00<?, ?it/s]

Total chunks: 1633
Embeddings: (1633, 768)
Per-patient embeddings ready for 400 patients


## Cell 5: Pipeline functions (modes 1/2/3)

In [8]:
PRED = {"indication":"is indicated for","contraindication":"is contraindicated in","side effect":"can cause","drug-drug interaction":"interacts with","synergistic interaction":"synergistically interacts with","target":"targets","enzyme":"is metabolized by","phenotype present":"presents with","associated with":"is associated with"}

def encode_query(q):
    e = qtok([q], truncation=True, padding=True, max_length=64, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        o = qenc(**e)
        v = F.normalize(o.last_hidden_state[:,0,:], dim=-1)
    return v.cpu().numpy()

def retrieve_patient(q, pid, k=4):
    if pid not in patient_embs_by_id: return []
    qv = encode_query(q)
    s = (patient_embs_by_id[pid] @ qv.T).flatten()
    chunks = patient_chunks_by_id[pid]
    idx = np.argsort(-s)[:k]
    return [{"score": float(s[i]), "chunk_id": chunks[i]["chunk_id"], "section": chunks[i]["section"], "text": chunks[i]["text"]} for i in idx]

def rerank(q, cands, k=3):
    if not cands: return []
    pairs = [[q, c["text"]] for c in cands]
    scores = []
    for i in range(0, len(pairs), 8):
        b = pairs[i:i+8]
        e = ctok(b, truncation=True, padding=True, max_length=512, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            l = cenc(**e).logits.squeeze(dim=-1)
        if l.dim()==0: scores.append(float(l))
        else: scores.extend(l.cpu().tolist())
    for c, s in zip(cands, scores): c["cross_encoder_score"] = float(s)
    return sorted(cands, key=lambda x: x["cross_encoder_score"], reverse=True)[:k]

def kg_for_drugs(drugs, k=4):
    triples, seen = [], set()
    for d in drugs:
        for v in [d.lower(), d.lower().split()[0]]:
            if v in name_to_triples:
                for rel, y, t in name_to_triples[v][:k]:
                    key = (v, rel, y)
                    if key in seen: continue
                    seen.add(key)
                    triples.append({"subject": v, "predicate": rel, "object": y, "object_type": t, "triple_text": f"{v} {rel} {y}"})
                break
    return triples

def kg_for_question(q, k=3):
    triples, seen = [], set()
    for ent in nlp(q).ents:
        e = ent.text.strip().lower()
        for v in (e, e.split()[0] if e.split() else "", e.replace("-"," ").split()[0] if e else ""):
            if not v or v in seen: continue
            if v in name_to_triples:
                for rel, y, t in name_to_triples[v][:k]:
                    key = (v, rel, y)
                    if key in seen: continue
                    seen.add(key)
                    triples.append({"subject": v, "predicate": rel, "object": y, "object_type": t, "triple_text": f"{v} {rel} {y}"})
    for w in re.findall(r"\b[a-z]{5,}\b", q.lower()):
        if len(triples) >= 12: break
        if w in name_to_triples:
            for rel, y, t in name_to_triples[w][:2]:
                key = (w, rel, y)
                if key in seen: continue
                seen.add(key)
                triples.append({"subject": w, "predicate": rel, "object": y, "object_type": t, "triple_text": f"{w} {rel} {y}"})
    return triples

def fmt_rx(p):
    m = p.get("medications", [])
    if not m: return "(no active prescriptions on file)"
    return "\n".join(f"- {x['drug']} {x['dosage']}, {x['frequency']}, for {x['indication']}" + (f" (Note: {x['notes']})" if x.get('notes') else "") for x in m)

def fmt_pctx(rk):
    return "=== PATIENT MEDICAL RECORD ===\n" + "\n".join(f"- {c['text']}" for c in rk) if rk else ""

def fmt_kg(tr):
    if not tr: return ""
    out = []
    for t in tr:
        ph = PRED.get(t['predicate'].lower(), t['predicate'])
        out.append(f"- {t['subject'].capitalize()} {ph} {t['object']}.")
    return "=== DRUG SAFETY INFORMATION (from validated drug databases) ===\n" + "\n".join(out)

def prompt_v1(q, p, ctx="", kg=""):
    parts = ["You are a helpful medical assistant answering questions for a patient about their",
             "prescriptions and health. Use the information provided to give a clear, accurate,",
             "and patient-friendly answer. If you do not have the information needed, say so",
             "honestly rather than guessing.", "", "=== ACTIVE PRESCRIPTIONS ===", fmt_rx(p), ""]
    if ctx: parts += [ctx, ""]
    if kg: parts += [kg, ""]
    parts += ["=== PATIENT QUESTION ===", q, "", "Provide a concise, helpful answer:"]
    return "\n".join(parts)

def prompt_v2_mode3(q, p, ctx, kg):
    return ("You are a helpful medical assistant answering questions for a patient about their\n"
            "prescriptions and health. Use the patient's prescriptions, medical record, and the\n"
            "drug safety information below to give a thoughtful, helpful answer.\n\n"
            "If the safety information directly addresses the question, share it in plain language.\n"
            "Recommending the patient confirm with their doctor is appropriate, but try to be\n"
            "informative first rather than only deferring. When the medical facts above clearly\n"
            "answer the question, lead with that information.\n\n"
            f"=== ACTIVE PRESCRIPTIONS ===\n{fmt_rx(p)}\n\n{ctx}\n\n{kg}\n\n"
            f"=== PATIENT QUESTION ===\n{q}\n\nProvide a concise, helpful answer:")

def query_gemma(prompt, temperature=0.0, max_tokens=512):
    try:
        r = requests.post(OLLAMA_URL, json={"model": GEMMA_MODEL, "prompt": prompt, "stream": False,
                                              "options": {"temperature": temperature, "num_predict": max_tokens, "seed": 42}}, timeout=300)
        r.raise_for_status()
        return r.json().get("response", "").strip()
    except Exception as e:
        return f"[ollama error: {e}]"

def mode_1(q, p):
    t = time.time()
    a = query_gemma(prompt_v1(q, p))
    return {"mode":1, "mode_name":"LLM_only", "answer":a, "latency_seconds":round(time.time()-t,2),
            "n_retrieved_chunks":0, "n_kg_triples":0, "retrieved_chunks":[], "kg_triples":[]}

def mode_2(q, p):
    t = time.time()
    cands = retrieve_patient(q, p["patient_id"], k=4)
    rk = rerank(q, cands, k=3)
    a = query_gemma(prompt_v1(q, p, ctx=fmt_pctx(rk)))
    return {"mode":2, "mode_name":"RAG", "answer":a, "latency_seconds":round(time.time()-t,2),
            "n_retrieved_chunks":len(rk), "n_kg_triples":0,
            "retrieved_chunks":[{"section":c["section"], "score":c.get("cross_encoder_score",c["score"]), "text":c["text"][:200]} for c in rk],
            "kg_triples":[]}

def mode_3(q, p):
    t = time.time()
    cands = retrieve_patient(q, p["patient_id"], k=4)
    rk = rerank(q, cands, k=3)
    drugs = [m["drug"] for m in p.get("medications", [])]
    triples = []
    seen = set()
    for tr in kg_for_drugs(drugs, k=4) + kg_for_question(q, k=3):
        key = (tr["subject"], tr["predicate"], tr["object"])
        if key in seen: continue
        seen.add(key)
        triples.append(tr)
    a = query_gemma(prompt_v2_mode3(q, p, fmt_pctx(rk), fmt_kg(triples)))
    return {"mode":3, "mode_name":"RAG_KG", "answer":a, "latency_seconds":round(time.time()-t,2),
            "n_retrieved_chunks":len(rk), "n_kg_triples":len(triples),
            "retrieved_chunks":[{"section":c["section"], "score":c.get("cross_encoder_score",c["score"]), "text":c["text"][:200]} for c in rk],
            "kg_triples":[t["triple_text"] for t in triples]}

print("Pipelines ready")

Pipelines ready


## Cell 6: Smoke test on 5 questions (verify everything works before the long run)

In [9]:
smoke = all_questions[:5]
t0 = time.time()
for q in smoke:
    p = patient_by_id[q["patient_id"]]
    r1 = mode_1(q["question"], p)
    r2 = mode_2(q["question"], p)
    r3 = mode_3(q["question"], p)
    print(f"{q['question_id']}  M1: {r1['latency_seconds']}s  M2: {r2['latency_seconds']}s  M3: {r3['latency_seconds']}s")
print(f"\nSmoke test: 5 questions × 3 modes = 15 LLM calls in {time.time()-t0:.1f}s sequential")
print("If this looks healthy, proceed to cell 7.")

Q001  M1: 84.75s  M2: 1.35s  M3: 1.37s
Q002  M1: 0.63s  M2: 0.74s  M3: 0.76s
Q003  M1: 1.34s  M2: 1.26s  M3: 1.48s
Q004  M1: 0.87s  M2: 1.14s  M3: 1.39s
Q005  M1: 1.2s  M2: 1.27s  M3: 0.96s

Smoke test: 5 questions × 3 modes = 15 LLM calls in 100.5s sequential
If this looks healthy, proceed to cell 7.


## Cell 7: PARALLEL run on all 2000 questions with checkpointing

Uses `ThreadPoolExecutor(max_workers=4)` so 4 questions are in-flight concurrently. Each question runs its 3 modes sequentially. Checkpoints to disk every 50 questions.

In [13]:
recs = load_jsonl(responses_path)
print(f"Records: {len(recs)}")
print(f"Sample question_ids: {[r['question_id'] for r in recs[:5]]}")
print(f"\nSample Mode 2 answer (Q001):")
sample = next((r for r in recs if r['question_id'] == 'Q001'), recs[0])
print(sample['mode_2']['answer'][:400])
print(f"\nLatency M1/M2/M3 (Q001): {sample['mode_1']['latency_seconds']}s / {sample['mode_2']['latency_seconds']}s / {sample['mode_3']['latency_seconds']}s")

Records: 2000
Sample question_ids: ['Q001', 'Q002', 'Q003', 'Q004', 'Q005']

Sample Mode 2 answer (Q001):
Metformin is a medication used to help manage Type 2 diabetes. It works by helping your body use insulin more effectively and by reducing the amount of sugar your liver produces.  It's important to take it with meals as directed to minimize any stomach upset you might experience. 


Let me know if you have any other questions about Metformin or your other medications!

Latency M1/M2/M3 (Q001): 84.01s / 1.46s / 1.68s


In [12]:
responses_path = DATA / "patient_portal_responses_v2_2000.jsonl"

# Resume support
done_ids = set()
if responses_path.exists():
    with open(responses_path) as f:
        for line in f:
            try: done_ids.add(json.loads(line)["question_id"])
            except Exception: pass
    print(f"Resuming: {len(done_ids)} done")

remaining = [q for q in all_questions if q["question_id"] not in done_ids]
print(f"To process: {len(remaining)}")

def run_one(q):
    """Run all 3 modes for a single question. Returns the full record."""
    p = patient_by_id[q["patient_id"]]
    try:
        r1 = mode_1(q["question"], p)
        r2 = mode_2(q["question"], p)
        r3 = mode_3(q["question"], p)
        return {
            "question_id": q["question_id"], "patient_id": q["patient_id"],
            "question": q["question"], "category": q["category"],
            "expected_mode": q["expected_mode"],
            "expected_mode_name": q.get("expected_mode_name", ""),
            "rationale": q.get("rationale", ""),
            "mode_1": r1, "mode_2": r2, "mode_3": r3,
        }
    except Exception as e:
        return {"question_id": q["question_id"], "error": str(e)}

CHECKPOINT_EVERY = 50
buffer = []
completed = 0
errors = 0
t0 = time.time()

with open(responses_path, "a") as fout:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(run_one, q): q for q in remaining}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"2000 q × 3 modes ({MAX_WORKERS}-way parallel)"):
            rec = fut.result()
            if rec.get("error"):
                errors += 1
                continue
            fout.write(json.dumps(rec) + "\n")
            completed += 1
            if completed % CHECKPOINT_EVERY == 0:
                fout.flush()
                elapsed = time.time() - t0
                rate = completed / elapsed
                eta = (len(futures) - completed) / rate if rate > 0 else 0
                print(f"\n[ckpt] {completed}/{len(futures)}  {rate:.2f} q/s  ETA {eta/60:.1f}min  errors {errors}")

print(f"\nDone. {completed} written, {errors} errors. {(time.time()-t0)/60:.1f} min total.")
print(f"Saved to {responses_path}")

Resuming: 2000 done
To process: 0


2000 q × 3 modes (4-way parallel): 0it [00:00, ?it/s]


Done. 0 written, 0 errors. 0.0 min total.
Saved to /content/drive/MyDrive/DL Project/synthetic_patients/patient_portal_responses_v2_2000.jsonl


In [14]:
n_m2_with_chunks = sum(1 for r in recs if r['mode_2'].get('n_retrieved_chunks', 0) > 0)
n_m3_with_chunks = sum(1 for r in recs if r['mode_3'].get('n_retrieved_chunks', 0) > 0)
n_m3_with_kg = sum(1 for r in recs if r['mode_3'].get('n_kg_triples', 0) > 0)
print(f"Mode 2 with patient chunks: {n_m2_with_chunks} / {len(recs)}")
print(f"Mode 3 with patient chunks: {n_m3_with_chunks} / {len(recs)}")
print(f"Mode 3 with KG triples:     {n_m3_with_kg} / {len(recs)}")

# Also peek at a Mode 2 record to see if retrieval actually fired
sample = recs[10]
print(f"\nSample (Q{sample['question_id']}): {len(sample['mode_2'].get('retrieved_chunks', []))} chunks")
print(f"Sample chunks: {[c.get('section') for c in sample['mode_2'].get('retrieved_chunks', [])]}")

Mode 2 with patient chunks: 2000 / 2000
Mode 3 with patient chunks: 2000 / 2000
Mode 3 with KG triples:     1961 / 2000

Sample (QQ011): 3 chunks
Sample chunks: ['lifestyle', 'allergies', 'recent_vitals']


## Cell 8: Score responses → ground_truth file

In [ ]:
PUNT_RE = re.compile(r"i can'?t give (you )?medical advice|i'?m sorry,? but i can'?t|i am not (a doctor|a medical|able to)|please (talk to|consult|speak to|discuss with) (your )?doctor|only your doctor can", re.IGNORECASE)

def is_pure_punt(answer):
    if not answer: return True
    if len(answer) < 250 and PUNT_RE.search(answer):
        useful = sum(1 for s in re.split(r"[.!?]+", answer) if len(s.strip()) >= 15 and not PUNT_RE.search(s))
        return useful <= 1
    return False

def cites_specific_value(a, p):
    al = a.lower()
    for k, v in p.get("recent_vitals", {}).items():
        if k == "date": continue
        if str(v).lower() in al: return True
    for x in p.get("allergies", []):
        if x.get("substance", "").lower() in al: return True
    for h in p.get("past_medical_history", []):
        if h.lower().split()[:3] and " ".join(h.lower().split()[:3]) in al: return True
    return False

def mentions_interaction(a):
    al = a.lower()
    return any(k in al for k in ["interact", "increase the levels", "decrease the levels", "bleeding risk", "cyp", "metabolism", "hyperkalemi", "potassium", "potentiate", "could interact", "may interact"])

def gives_safety_advice(a):
    al = a.lower()
    return any(re.search(p, al) for p in [r"avoid", r"don'?t (take|drink|combine)", r"safe to take", r"can take .* with", r"should not", r"contraindicated", r"caution", r"may cause", r"could interact"])

def is_informative_diplomatic(a, p):
    if len(a) < 200: return False
    al = a.lower()
    has_specific = mentions_interaction(a)
    if not has_specific:
        for m in p.get("medications", []):
            if m["drug"].split()[0].lower() in al and len(m["drug"].split()[0]) > 3:
                has_specific = True; break
    return has_specific and sum(1 for s in re.split(r"[.!?]+", a) if len(s.strip()) >= 25 and not PUNT_RE.search(s.lower())) >= 2

def score_mode(rec, p, expected, rationale):
    a = rec["answer"]
    if not a or len(a) < 30: return 0, "empty/short"
    if is_pure_punt(a): return 0, "punted"
    m = rec["mode"]
    if expected == "LLM_only": return 1, "generic OK for any mode"
    if expected == "RAG":
        if m == 1:
            r = rationale.lower()
            if any(k in r for k in ["notes", "prescription", "indication"]) and any(x["drug"].lower() in a.lower() for x in p.get("medications", [])):
                return 1, "answer in prescription text"
            if any(k in r for k in ["vital", "actual value", "history", "patient context"]):
                return (1, "got specific value") if cites_specific_value(a, p) else (0, "needed patient data")
            if "lifestyle" in r: return 0, "needed lifestyle context"
            return (1, "reasonable") if len(a) > 100 else (0, "too generic")
        return 1, "patient context applied"
    if expected == "RAG_KG":
        if mentions_interaction(a) and gives_safety_advice(a): return 1, "interaction + concrete advice"
        if gives_safety_advice(a): return 1, "concrete safety advice"
        if is_informative_diplomatic(a, p): return 1, "diplomatic but informative"
        return 0, "missed safety/interaction"
    return 0, "unknown"

def score_record(r, p):
    exp = r["expected_mode_name"]
    rt = r["rationale"]
    s1, w1 = score_mode(r["mode_1"], p, exp, rt)
    s2, w2 = score_mode(r["mode_2"], p, exp, rt)
    s3, w3 = score_mode(r["mode_3"], p, exp, rt)
    winner = 1 if s1 else (2 if s2 else (3 if s3 else 0))
    return {"question_id": r["question_id"], "patient_id": r["patient_id"], "category": r["category"],
            "expected_mode": r["expected_mode"], "expected_mode_name": r["expected_mode_name"],
            "mode_1_correct": s1, "mode_1_rationale": w1,
            "mode_2_correct": s2, "mode_2_rationale": w2,
            "mode_3_correct": s3, "mode_3_rationale": w3,
            "winning_mode": winner}

responses_all = load_jsonl(responses_path)
print(f"Loaded {len(responses_all)} responses")

gt = []
for r in tqdm(responses_all, desc="scoring"):
    p = patient_by_id[r["patient_id"]]
    gt.append(score_record(r, p))

gt_path = DATA / "patient_portal_ground_truth_v2_2000.jsonl"
with open(gt_path, "w") as f:
    for g in gt: f.write(json.dumps(g) + "\n")
print(f"\nSaved {gt_path}")

from collections import Counter
n = len(gt)
print(f"\nMode 1: {sum(g['mode_1_correct'] for g in gt)/n*100:.1f}%")
print(f"Mode 2: {sum(g['mode_2_correct'] for g in gt)/n*100:.1f}%")
print(f"Mode 3: {sum(g['mode_3_correct'] for g in gt)/n*100:.1f}%")
print(f"Oracle: {sum(1 for g in gt if g['mode_1_correct'] or g['mode_2_correct'] or g['mode_3_correct'])/n*100:.1f}%")
print("\nWinning mode distribution:", Counter(g['winning_mode'] for g in gt).most_common())
print("\nDownload these two files from Drive to your Mac:")
print(f"  {responses_path}")
print(f"  {gt_path}")
print("\nThen run scripts/retrain_router_v3.py locally — it will pick up the real winning_mode labels for all 2000.")